# NOOTEBOOK DE CONFECCION DE SOLAPAMINETO DE BANDAS

## Aqu
ASD}

In [ ]:
"""
Procesar Sentinel-2 descargado en estructura tipo:
E:\Silos\Base de datos\s2_aws_downloads_structured\tiles\20\H\NJ\2020\6\3\1\R10m\B02.jp2  (ejemplo)

Para cada escena en años 2020-2025 y meses 6-8:
 - busca B02,B03,B04,B08 (10m) dentro de R10m
 - busca SCL.jp2 (20m) en la escena (por ejemplo en R20m o carpeta 20m)
 - remuestrea SCL a 10m (nearest)
 - crea máscara "clear" = SCL in [4,5,6]
 - crea compuesto (B02,B03,B04,B08) y aplica máscara
 - busca el mayor componente conectado 100% claro y recorta la imagen al bbox de ese componente
 - guarda: compuesto completo (con nodata fuera de máscara), máscara, y recorte del bbox
"""

In [ ]:
import os
import rasterio
from rasterio.enums import Resampling
import numpy as np
from rasterio import warp # <--- LÍNEA AÑADIDA: Importamos el módulo warp explícitamente

def procesar_imagenes_sentinel(ruta_base, ruta_salida):
    """
    Procesa imágenes Sentinel-2 para crear compuestos de 4 bandas sin nubes.

    Args:
        ruta_base (str): La ruta principal donde se encuentran las carpetas de tiles.
        ruta_salida (str): La carpeta donde se guardarán los archivos procesados.
    """
    os.makedirs(ruta_salida, exist_ok=True)
    print(f"La salida se guardará en: {ruta_salida}")

    valores_scl_validos = [4, 5, 6, 7, 11]
    bandas_10m_a_usar = ['B04', 'B03', 'B02', 'B08']

    for root, dirs, files in os.walk(ruta_base):
        if root.endswith("R10m"):
            print(f"\nProcesando imagen en: {root}")
            
            ruta_scl_20m = root.replace("R10m", "R20m")
            archivo_scl = os.path.join(ruta_scl_20m, "SCL.jp2")
            
            rutas_bandas = {banda: os.path.join(root, f"{banda}.jp2") for banda in bandas_10m_a_usar}

            if not os.path.exists(archivo_scl):
                print(f"  -> ADVERTENCIA: No se encontró el archivo SCL en {ruta_scl_20m}. Omitiendo.")
                continue
            if not all(os.path.exists(p) for p in rutas_bandas.values()):
                print(f"  -> ADVERTENCIA: Faltan una o más bandas de 10m en {root}. Omitiendo.")
                continue

            try:
                with rasterio.open(archivo_scl) as src_scl:
                    scl_data = src_scl.read(1)
                    perfil_scl = src_scl.profile
                    mascara_20m = np.isin(scl_data, valores_scl_validos)

                with rasterio.open(rutas_bandas['B04']) as src_10m_sample:
                    perfil_10m = src_10m_sample.profile
                    forma_destino = src_10m_sample.shape

                mascara_10m = np.empty(forma_destino, dtype=np.uint8)
                
                # LÍNEA CORREGIDA: Usamos 'warp.reproject' en lugar de 'rasterio.warp.reproject'
                warp.reproject(
                    source=mascara_20m.astype(np.uint8),
                    destination=mascara_10m,
                    src_transform=perfil_scl['transform'],
                    src_crs=perfil_scl['crs'],
                    dst_transform=perfil_10m['transform'],
                    dst_crs=perfil_10m['crs'],
                    resampling=Resampling.nearest
                )
                
                mascara_10m_bool = mascara_10m.astype(bool)

                stack_10m = []
                for banda in bandas_10m_a_usar:
                    with rasterio.open(rutas_bandas[banda]) as src_banda:
                        stack_10m.append(src_banda.read(1))
                
                array_stack = np.array(stack_10m, dtype=perfil_10m['dtype'])

                valor_nodata = 0
                array_stack[:, ~mascara_10m_bool] = valor_nodata

                partes_ruta = root.replace(ruta_base, '').strip(os.sep).split(os.sep)
                nombre_archivo = f"{'_'.join(partes_ruta)}.tif"
                ruta_archivo_salida = os.path.join(ruta_salida, nombre_archivo)

                perfil_salida = perfil_10m.copy()
                perfil_salida.update({
                    'driver': 'GTiff',
                    'count': len(bandas_10m_a_usar),
                    'nodata': valor_nodata,
                    'compress': 'lzw'
                })

                with rasterio.open(ruta_archivo_salida, 'w', **perfil_salida) as dst:
                    dst.write(array_stack)
                
                print(f"  -> ÉXITO: Imagen procesada y guardada en {ruta_archivo_salida}")

            except Exception as e:
                print(f"  -> ERROR procesando {root}: {e}")

# --- CONFIGURACIÓN ---
RUTA_DATOS_AWS = r"E:\Silos\Base de datos\s2_aws_downloads_structured\tiles"
RUTA_SALIDA_PROCESADO = r"E:\Silos\Base de datos\procesado_sin_nubes"

# --- EJECUCIÓN ---
procesar_imagenes_sentinel(RUTA_DATOS_AWS, RUTA_SALIDA_PROCESADO)

print("\n--- Proceso finalizado ---")

La salida se guardará en: E:\Silos\Base de datos\procesado_sin_nubes

Procesando imagen en: E:\Silos\Base de datos\s2_aws_downloads_structured\tiles\20\H\NH\2020\6\10\0\R10m
  -> ÉXITO: Imagen procesada y guardada en E:\Silos\Base de datos\procesado_sin_nubes\20_H_NH_2020_6_10_0_R10m.tif

Procesando imagen en: E:\Silos\Base de datos\s2_aws_downloads_structured\tiles\20\H\NH\2020\6\10\1\R10m
  -> ÉXITO: Imagen procesada y guardada en E:\Silos\Base de datos\procesado_sin_nubes\20_H_NH_2020_6_10_1_R10m.tif

Procesando imagen en: E:\Silos\Base de datos\s2_aws_downloads_structured\tiles\20\H\NH\2020\6\13\0\R10m
  -> ÉXITO: Imagen procesada y guardada en E:\Silos\Base de datos\procesado_sin_nubes\20_H_NH_2020_6_13_0_R10m.tif

Procesando imagen en: E:\Silos\Base de datos\s2_aws_downloads_structured\tiles\20\H\NH\2020\6\13\1\R10m
  -> ÉXITO: Imagen procesada y guardada en E:\Silos\Base de datos\procesado_sin_nubes\20_H_NH_2020_6_13_1_R10m.tif

Procesando imagen en: E:\Silos\Base de datos\s2_aw